# Raw JSON/JSONL Analysis

**Author:** Databricks Ingestion Team  
**Version:** 1.0  
**Last Updated:** 2026-03-31

This notebook analyzes raw JSON and JSONL files, infers schema, and provides insights into nested structures. It is Databricks-ready and follows production best practices.

---

**Parameters:**
- `input_path`: Path to the raw JSON/JSONL file (set via widget)
- `sample_size`: Number of records to sample for schema inference

---

**Instructions:**
1. Set the input path and sample size using the widgets below.
2. Run all cells to analyze the file and view schema/structure insights.

---

**Notebook Metadata:**
- **Tested on:** Databricks Runtime 12.2 LTS
- **Contact:** data-eng@databricks.com


In [ ]:
# Databricks widgets for parameterization
import json
import pyspark.sql.functions as F
from pyspark.sql.types import StructType
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("json_analysis")

dbutils.widgets.text("input_path", "", "Input JSON/JSONL Path")
dbutils.widgets.text("sample_size", "1000", "Sample Size")
input_path = dbutils.widgets.get("input_path")
sample_size = int(dbutils.widgets.get("sample_size"))

# Read sample data
try:
    df = spark.read.json(input_path, multiLine=True)
    df_sample = df.limit(sample_size)
    display(df_sample)
    logger.info(f"Loaded {df_sample.count()} records from {input_path}")
except Exception as e:
    logger.error(f"Error reading JSON/JSONL: {e}")
    df_sample = None

# Infer schema and show nested structure
if df_sample is not None:
    print("\n--- Inferred Schema ---\n")
    df_sample.printSchema()
    print("\n--- Sample Data ---\n")
    display(df_sample)
    # Show nested fields
    def print_nested_fields(schema, prefix=""):
        for field in schema.fields:
            if isinstance(field.dataType, StructType):
                print(f"{prefix}{field.name} (struct):")
                print_nested_fields(field.dataType, prefix + "  ")
            else:
                print(f"{prefix}{field.name}: {field.dataType}")
    print("\n--- Nested Structure ---\n")
    print_nested_fields(df_sample.schema)
else:
    print("No data to analyze.")

# Example inline comment for maintainability
# This notebook is designed for robust schema inference and nested structure analysis in Databricks production workflows.


---

## Validation & Testing

Below, we validate the notebook logic with a simple assertion to ensure data was loaded. Add more tests as needed for your data.


In [ ]:
# Simple validation
if df_sample is not None:
    assert df_sample.count() > 0, "No records loaded from input file."
    print("Validation passed: Data loaded.")
else:
    print("Validation failed: No data loaded.")


---

## Resource Cleanup

Unpersist DataFrames and clean up resources if needed to avoid memory issues in production workflows.

In [ ]:
# Resource cleanup
if 'df_sample' in locals() and df_sample is not None:
    df_sample.unpersist(blocking=True)
    print("Resources cleaned up.")
